← [Redes neuronales](06-redes-neuronales.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Cambio de dominio](08-cambio-de-dominio.ipynb) →

# 07 · Transfer learning y MobileNet



## El problema del tamaño

Entrenar una CNN desde cero necesita millones de imágenes. Reci tenía 26.301.
Con tan pocas, una red grande **memoriza** el conjunto de entrenamiento en vez
de aprender a distinguir materiales: acierta todo lo que ya vio y falla en lo
nuevo. Eso se llama **sobreajuste** (*overfitting*).



## Transfer learning

La solución aprovecha algo del documento anterior: las primeras capas de
cualquier CNN aprenden bordes, texturas y brillos. **Eso es igual para
cualquier tarea visual** — un detector de bordes sirve tanto para reconocer
perros como botellas.

Entonces:

```
1. tomar una red ya entrenada con millones de fotos genéricas (ImageNet)
2. congelar sus capas: ya saben ver
3. reemplazar solo la última capa por una nueva de 2 salidas
4. entrenar esa capa con las fotos propias
```

La red no aprende a ver desde cero: aprende a **traducir** lo que ya sabe ver al
vocabulario del problema. Por eso 26.000 fotos alcanzan.

**ImageNet** es el dataset de referencia: ~1,2 millones de fotos en 1.000
categorías. Ninguna es "plástico" o "vidrio", y no importa — lo que se hereda
es la capacidad de extraer características visuales.



## Las dos fases

En Reci el entrenamiento tiene dos etapas, visibles en `entrenador.py`:

| Fase | Qué se entrena | Tasa de aprendizaje | Resultado histórico |
| --- | --- | --- | --- |
| 1 · Cabeza | Solo la capa final | `1e-3` (alta) | 90,12 % |
| 2 · Ajuste fino | Últimas capas del backbone + cabeza | `1e-5` (100× menor) | 98,43 % |

La lógica del orden: si se descongelara el backbone desde el principio, la
capa final —que empieza con valores aleatorios— produciría errores enormes que
destruirían los pesos buenos de ImageNet. Primero se estabiliza la cabeza,
después se afina el resto **con pasos muy pequeños** para no borrar lo heredado.

Un detalle del código que responde a lo mismo: las capas de `BatchNormalization`
se mantienen congeladas durante la fase 2. Con lotes pequeños, actualizar sus
estadísticas desestabiliza el ajuste.



## MobileNetV2

De todas las CNN disponibles, Reci usó durante todo el proyecto
**MobileNetV2**, diseñada para dispositivos con recursos limitados. Es la
arquitectura sobre la que se explica el resto de la serie.

Su idea central es la **convolución separable**: en vez de mirar todos los
canales de color a la vez, primero procesa cada canal por separado
(*depthwise*) y después los combina (*pointwise*). El resultado es parecido con
una fracción de las operaciones.

```
entrada → expansión 1×1 → depthwise 3×3 → proyección 1×1
                                          + conexión residual
```

La ficha del MobileNetV2 que estuvo desplegado hasta el 12 de agosto de
2026:

| Dato | Valor |
| --- | --- |
| Entrada | 224 × 224 × 3, valores crudos de 0 a 255 |
| Cabeza | GlobalAveragePooling → Dropout → Dense(2, softmax) |
| Salidas | `plastico`, `vidrio` — solo esas dos |
| Parámetros | ≈ 2,36 millones |
| Formato | TensorFlow Lite, ~8,5 MB |
| Corrida | `run_20260721_2129` |

> **Nota de estado.** Desde el 12 de agosto de 2026 el artefacto activo es
> un **MobileNetV3-Large** cuantizado a int8, salido del experimento
> comparativo del 9 de agosto. Ese cambio tiene una verificación pendiente
> que se explica en [Del modelo al robot](12-del-modelo-al-robot.ipynb);
> el estado técnico vigente vive en
> [`ia/vision-service/model/README.md`](../../ia/vision-service/model/README.md).
> Los conceptos de este documento —transfer learning, convolución
> separable, las dos fases— valen igual para las tres arquitecturas.



## Preprocesamiento: un detalle que importa

Cada arquitectura espera sus valores de entrada en un rango distinto:

| Arquitectura | Rango esperado |
| --- | --- |
| MobileNetV2 | de −1 a 1 |
| EfficientNet-B0 | de 0 a 255 (normaliza internamente) |
| MobileNetV3-Large | de 0 a 255 (con `include_preprocessing=True`) |

Si se alimenta un modelo con el rango equivocado, **no da error**: da
predicciones sin sentido. Es un fallo silencioso, de los peores.

Por eso los scripts de entrenamiento de Reci **hornean el preprocesamiento
dentro del modelo exportado**. El `.tflite` recibe siempre píxeles crudos de 0 a
255 y cada arquitectura resuelve su normalización por dentro. Así los tres
candidatos son intercambiables sin tocar `vision/local_model.py`.



## De dónde vino y a dónde va

Este modelo lo entrenó Axel Hernández en el repositorio RECI2 y se portó
aquí como clasificador independiente. Su validación fue **98,43 %**.

Ese número es engañoso, y entender por qué es el tema del siguiente documento —
probablemente el más importante de la serie.

---

← [Redes neuronales](06-redes-neuronales.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Cambio de dominio](08-cambio-de-dominio.ipynb) →
